# Notebook 01: Bronze Layer — Ingestion & Raw Preservation

This notebook handles loading raw data from the **Online Retail II** dataset (Excel format) into the **Bronze Layer** using **Polars** and saving it in **Parquet** format.

---

## 1. Design Principles & Goals
* **Immutability:** Preserves all raw rows and columns without applying any premature filtering, dropping missing values, or altering records.
* **Schema Overrides:** Forces identifier columns (`Invoice`, `StockCode`, `Customer ID`) to `Utf8` (String) type to prevent losing prefix letters (e.g., `C` for cancellations) or leading zeros.
* **High Performance Format:** Converts raw Excel sheets into Parquet files in `data/1_bronze/`, optimizing storage space and significantly boosting reading speed for downstream Silver layer pipelines.

---

## 2. Data Ingestion Pipeline

In [ ]:
from pathlib import Path
import polars as pl
from loguru import logger


In [ ]:
BASE_DIR = Path("..")
RAW_EXCEL_PATH = BASE_DIR / "data" / "raw" / "online_retail_II.xlsx"
BRONZE_DIR = BASE_DIR / "data" / "1_bronze"
BRONZE_DIR.mkdir(parents=True, exist_ok=True)

logger.info("Nạp nguyên bản dữ liệu Excel (Khóa cứng kiểu String cho Invoice & StockCode)...")

2026-11-06 10:58:29.448 | INFO     | __main__:<module>:6 - Nạp nguyên bản dữ liệu Excel (Khóa cứng kiểu String cho Invoice & StockCode)...


In [3]:
# Dùng schema_overrides để giữ nguyên chữ 'C' ở Invoice không bị biến thành NULL
df_0910_raw = pl.read_excel(
    RAW_EXCEL_PATH, 
    sheet_name="Year 2009-2010",
    schema_overrides={"Invoice": pl.Utf8, "StockCode": pl.Utf8, "Customer ID": pl.Utf8}
)

df_1011_raw = pl.read_excel(
    RAW_EXCEL_PATH, 
    sheet_name="Year 2010-2011",
    schema_overrides={"Invoice": pl.Utf8, "StockCode": pl.Utf8, "Customer ID": pl.Utf8}
)

# Ghi lại file Parquet tầng Bronze chuẩn
df_0910_raw.write_parquet(BRONZE_DIR / "sheet_2009_2010_raw.parquet")
df_1011_raw.write_parquet(BRONZE_DIR / "sheet_2010_2011_raw.parquet")

logger.success(f"Bronze Sheet 1: {len(df_0910_raw):,} dòng")
logger.success(f"Bronze Sheet 2: {len(df_1011_raw):,} dòng")

2026-11-06 11:00:45.220 | SUCCESS  | __main__:<module>:18 - Bronze Sheet 1: 525,461 dòng
2026-11-06 11:00:45.222 | SUCCESS  | __main__:<module>:19 - Bronze Sheet 2: 541,910 dòng


In [6]:
display(df_0910_raw.head(5))

Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
str,str,str,i64,datetime[ms],f64,str,str
"""489434""","""85048""","""15CM CHRISTMAS GLASS BALL 20 L…",12,2009-12-01 07:45:00,6.95,"""13085""","""United Kingdom"""
"""489434""","""79323P""","""PINK CHERRY LIGHTS""",12,2009-12-01 07:45:00,6.75,"""13085""","""United Kingdom"""
"""489434""","""79323W""",""" WHITE CHERRY LIGHTS""",12,2009-12-01 07:45:00,6.75,"""13085""","""United Kingdom"""
"""489434""","""22041""","""RECORD FRAME 7"" SINGLE SIZE """,48,2009-12-01 07:45:00,2.1,"""13085""","""United Kingdom"""
"""489434""","""21232""","""STRAWBERRY CERAMIC TRINKET BOX""",24,2009-12-01 07:45:00,1.25,"""13085""","""United Kingdom"""
